# Qwen Binary-Classifier Fine-Tuning (DBpedia)

Fine-tunes `Qwen/Qwen3-Embedding-0.6B` with **LoRA + 4-bit** on the balanced `(anchor, query, label)` pairs produced by `01_create_qwen_binary_dataset.ipynb`.

- Loss      : **`CosineSimilarityLoss`**  (label=1 → cos→1, label=0 → cos→0)
- Evaluator : **`BinaryClassificationEvaluator`** on val pairs (per-pair accuracy / F1 / MCC / AP)
- Input     : `dataset/qwen_binary/dbpedia/{train,val}.json`
- Output    : `checkpoints/qwen_dbpedia_binary/final_merged/`  (LoRA merged into base → portable SentenceTransformer)


In [1]:
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import json
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

from sentence_transformers import (
    SentenceTransformer,
    InputExample,
    losses,
    models,
)
from sentence_transformers.evaluation import BinaryClassificationEvaluator

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU  : NVIDIA GeForce RTX 3090
VRAM : 25.3 GB


In [2]:
BASE_MODEL          = "Qwen/Qwen3-Embedding-0.6B"
MAX_SEQ_LENGTH      = 512

LORA_R              = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

BATCH_SIZE          = 8
NUM_EPOCHS          = 5
WARMUP_STEPS        = 100
LR                  = 1e-4
SEED                = 42

USE_4BIT            = False

DATASET_NAME        = "dbpedia"


def find_repo_root(start=None):
    """Walk up from `start` until a dir holding dataset/qwen_binary/<ds>/ is found."""
    start = Path(start or Path.cwd()).resolve()
    target = Path("qwen_finetune_dataset") / DATASET_NAME
    for p in [start, *start.parents]:
        if (p / target).is_dir():
            return p
    raise FileNotFoundError(
        f"Could not locate repo root containing {target} starting from: {start}.\n"
        f"Run 01_create_qwen_binary_dataset.ipynb first."
    )


env_root   = os.environ.get("SPARQL_EMB_ROOT")
REPO_ROOT  = Path(env_root).resolve() if env_root else find_repo_root()
DATA_DIR   = REPO_ROOT / "qwen_finetune_dataset" / DATASET_NAME 
OUTPUT_DIR = REPO_ROOT / "checkpoints" / f"qwen_{DATASET_NAME}_binary"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.json"
VAL_PATH   = DATA_DIR / "val.json"
assert TRAIN_PATH.exists(), f"Missing {TRAIN_PATH}. Run 01_create_qwen_binary_dataset.ipynb first."
assert VAL_PATH.exists(),   f"Missing {VAL_PATH}. Run 01_create_qwen_binary_dataset.ipynb first."

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Repo root  : {REPO_ROOT}")
print(f"Data dir   : {DATA_DIR}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"Base model : {BASE_MODEL}")
print(f"LoRA       : r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGET_MODULES}")
print(f"Batch/ep   : {BATCH_SIZE} / {NUM_EPOCHS}")
print(f"LR/warmup  : {LR} / {WARMUP_STEPS}")
print(f"4-bit      : {USE_4BIT}")

Repo root  : /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier
Data dir   : /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/qwen_finetune_dataset/dbpedia
Output dir : /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/checkpoints/qwen_dbpedia_binary
Base model : Qwen/Qwen3-Embedding-0.6B
LoRA       : r=16, alpha=32, targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
Batch/ep   : 8 / 5
LR/warmup  : 0.0001 / 100
4-bit      : False


In [3]:
with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(VAL_PATH, "r", encoding="utf-8") as f:
    val_data = json.load(f)

train_examples = [
    InputExample(texts=[r["anchor_query"], r["query"]], label=float(r["label"]))
    for r in train_data
]

val_s1     = [r["anchor_query"] for r in val_data]
val_s2     = [r["query"]        for r in val_data]
val_labels = [int(r["label"])   for r in val_data]

train_pos = sum(1 for r in train_data if int(r["label"]) == 1)
train_neg = len(train_data) - train_pos
val_pos   = sum(1 for l in val_labels if l == 1)
val_neg   = len(val_labels) - val_pos

print(f"Train pairs : {len(train_examples)}  (pos={train_pos}, neg={train_neg})")
print(f"Val   pairs : {len(val_labels)}  (pos={val_pos}, neg={val_neg})")

Train pairs : 62954  (pos=31477, neg=31477)
Val   pairs : 8194  (pos=4097, neg=4097)


In [4]:
evaluator = BinaryClassificationEvaluator(
    sentences1        = val_s1,
    sentences2        = val_s2,
    labels            = val_labels,
    name              = f"{DATASET_NAME}_binary_val",
    batch_size        = 32,
    show_progress_bar = False,
)
print(f"Evaluator ready on {len(val_labels)} val pairs.")

Evaluator ready on 8194 val pairs.


In [5]:
def build_qwen_st_with_lora(base_model_name, use_4bit=True, max_seq_length=512):
    """Load Qwen3-Embedding once, wrap with LoRA, expose as a SentenceTransformer
    with last-token pooling."""
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_name, trust_remote_code=True, padding_side="left"
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = None
    if use_4bit and torch.cuda.is_available():
        bnb_config = BitsAndBytesConfig(
            load_in_4bit              = True,
            bnb_4bit_quant_type       = "nf4",
            bnb_4bit_compute_dtype    = torch.bfloat16,
            bnb_4bit_use_double_quant = True,
        )

    base_model = AutoModel.from_pretrained(
        base_model_name,
        quantization_config = bnb_config,
        torch_dtype         = None,
        trust_remote_code   = True,
        device_map          = None,
    )

    lora_cfg = LoraConfig(
        r              = LORA_R,
        lora_alpha     = LORA_ALPHA,
        lora_dropout   = LORA_DROPOUT,
        target_modules = LORA_TARGET_MODULES,
        bias           = "none",
        task_type      = TaskType.FEATURE_EXTRACTION,
    )
    peft_base = get_peft_model(base_model, lora_cfg)
    peft_base.print_trainable_parameters()

    word_module = models.Transformer(
        base_model_name,
        max_seq_length = max_seq_length,
        tokenizer_args = {"trust_remote_code": True, "padding_side": "left"},
        model_args     = {"trust_remote_code": True},
    )
    word_module.auto_model = peft_base

    pooling = models.Pooling(
        word_module.get_word_embedding_dimension(),
        pooling_mode_lasttoken   = True,
        pooling_mode_mean_tokens = False,
    )

    return SentenceTransformer(modules=[word_module, pooling], device=device)


model = build_qwen_st_with_lora(BASE_MODEL, use_4bit=USE_4BIT, max_seq_length=MAX_SEQ_LENGTH)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5909.67it/s]


trainable params: 10,092,544 || all params: 605,869,056 || trainable%: 1.6658


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 4411.66it/s]


In [6]:
class SafeCosineSimilarityLoss(losses.CosineSimilarityLoss):
    """CosineSimilarityLoss that casts labels to the embedding dtype.

    Fixes `RuntimeError: Found dtype Float but expected Half/BFloat16` that occurs
    when the base model runs in fp16/bf16 but `InputExample(label=float(...))`
    arrives as fp32 — MSELoss refuses to mix dtypes in the backward pass.
    """

    def forward(self, sentence_features, labels):
        emb_a = self.model(sentence_features[0])["sentence_embedding"]
        emb_b = self.model(sentence_features[1])["sentence_embedding"]
        out   = self.cos_score_transformation(torch.cosine_similarity(emb_a, emb_b))
        return self.loss_fct(out, labels.to(out.dtype).view(-1))

train_loader = DataLoader(
    train_examples,
    shuffle    = True,
    batch_size = BATCH_SIZE,
    drop_last  = True,
)

train_loss = SafeCosineSimilarityLoss(model)

steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * NUM_EPOCHS
print(f"Steps / epoch : {steps_per_epoch}")
print(f"Total steps   : {total_steps}")
print(f"Warmup steps  : {WARMUP_STEPS}")

model.fit(
    train_objectives      = [(train_loader, train_loss)],
    evaluator             = evaluator,
    epochs                = NUM_EPOCHS,
    warmup_steps          = WARMUP_STEPS,
    optimizer_params      = {"lr": LR},
    output_path           = str(OUTPUT_DIR),
    save_best_model       = True,
    evaluation_steps      = steps_per_epoch,
    show_progress_bar     = True,
    use_amp               = False,
    checkpoint_path       = str(OUTPUT_DIR / "checkpoints"),
    checkpoint_save_steps = steps_per_epoch,
)
print("\n\u2705 Training done.")

Steps / epoch : 7869
Total steps   : 39345
Warmup steps  : 100


Step,Training Loss,Validation Loss,Dbpedia Binary Val Cosine Accuracy,Dbpedia Binary Val Cosine Accuracy Threshold,Dbpedia Binary Val Cosine F1,Dbpedia Binary Val Cosine F1 Threshold,Dbpedia Binary Val Cosine Precision,Dbpedia Binary Val Cosine Recall,Dbpedia Binary Val Cosine Ap,Dbpedia Binary Val Cosine Mcc
7869,0.003179,No log,0.999146,0.432412,0.999146,0.432412,0.999268,0.999024,0.999899,0.998291
15738,0.001236,No log,0.999146,0.606954,0.999146,0.261933,0.998780,0.999512,0.999994,0.998292
23607,0.000996,No log,0.998169,0.217810,0.998173,0.208437,0.996352,1.000000,0.999983,0.996345
31476,0.000735,No log,0.997925,0.190929,0.997930,0.190929,0.995868,1.000000,0.999978,0.995859
39345,0.000635,No log,0.997315,0.666586,0.997322,0.173903,0.994659,1.000000,0.999970,0.994645


/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/sentence_transformers/util/tensor.py:28: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  a = torch.tensor(a)



✅ Training done.


In [7]:
final_merged_dir = OUTPUT_DIR / "final_merged"
final_merged_dir.mkdir(parents=True, exist_ok=True)

word_module = model[0]
inner       = word_module.auto_model


def find_peft(module):
    if isinstance(module, PeftModel):
        return module
    for child in module.modules():
        if isinstance(child, PeftModel):
            return child
    return None


peft_model = find_peft(inner)
if peft_model is not None:
    print("Merging LoRA weights into base...")
    word_module.auto_model = peft_model.merge_and_unload()
else:
    print("No PeftModel wrapper detected; saving current weights as-is.")

model.save(str(final_merged_dir))
print(f"\u2705 Saved merged model \u2192 {final_merged_dir}")

Merging LoRA weights into base...


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.82s/it]


✅ Saved merged model → /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/checkpoints/qwen_dbpedia_binary/final_merged


In [8]:
print("Reloading merged model to verify clean load...")
reloaded = SentenceTransformer(str(final_merged_dir))
scores_reloaded = evaluator(reloaded)

print("\nBinary classification metrics on val (reloaded):")
for k, v in scores_reloaded.items():
    print(f"  {k}: {v:.4f}")

del reloaded
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Reloading merged model to verify clean load...


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5221.22it/s]



Binary classification metrics on val (reloaded):
  dbpedia_binary_val_cosine_accuracy: 0.9973
  dbpedia_binary_val_cosine_accuracy_threshold: 0.6617
  dbpedia_binary_val_cosine_f1: 0.9973
  dbpedia_binary_val_cosine_f1_threshold: 0.1772
  dbpedia_binary_val_cosine_precision: 0.9947
  dbpedia_binary_val_cosine_recall: 1.0000
  dbpedia_binary_val_cosine_ap: 1.0000
  dbpedia_binary_val_cosine_mcc: 0.9946


### Next step

This fine-tune is aligned with the binary-pair task.
To measure **retrieval** performance (and compare head-to-head with MNRL / SPARQLCE),
point your retrieval-evaluation notebook at:

```
checkpoints/qwen_dbpedia_binary/final_merged
```

Expect this baseline to under-perform the MNRL fine-tune on MRR / nDCG / Recall — that's the
intended narrative for the paper (binary cosine FT learns a point-wise similarity
that does not transfer to ranking). If it instead matches or beats MNRL, re-inspect the
triplet sampling in `01_qwen_finetune_mnrl_dbpedia.ipynb`.